# 気象観測データを DOI で取り出す（UTForestDataFetcher）

東京大学演習林の気象月報（1 か月・1 地点で 1 ファイル、ファイルごとに DOI）を、版を固定して取り出し、つなげて可視化します。

1. 準備（インストールと API の宛先）
2. 一覧（Catalog）
3. DOI の解決（DoiResolver）
4. データの取得と可視化（FetchData、`DataFrame.plot`）
5. 版と更新の確認（as_of、CheckUpdates）
6. 引用（Cite）

MyBinder、Google Colab、JupyterLite（ブラウザの中の Python）のどれでも動きます。今の DOI は mock（`10.xxxxx/...`）で、本物の DOI ではありません。

## 1. 準備

### インストール

JupyterLite ではサイトに同梱した wheel を、ほかではこの repo から入れます。

In [ ]:
import sys

if sys.platform == "emscripten":  # JupyterLite (Pyodide)
    %pip install -q UTForestDataFetcher matplotlib
else:
    try:
        import UTForestDataFetcher  # noqa: F401  (already installed, e.g. on MyBinder)
    except ImportError:
        %pip install -q "UTForestDataFetcher[plot] @ git+https://github.com/esehiyoko/UTForestDataFetcher.git"

### API の宛先

API の宛先はまだ内部のものなので、この notebook には書きません。次の順に探します。

1. 下の `API_BASE_URL` に書いた値（共有する前に消してください）
2. 環境変数 `UTFOREST_API_BASE_URL`
3. Google Colab の secret `UTFOREST_API_BASE_URL`（左の鍵のアイコン）
4. どれもなければ、入力を求めます

JupyterLite から呼ぶときは、API がそのページの origin を CORS で許している必要があります。

In [ ]:
import os
import UTForestDataFetcher
from UTForestDataFetcher import Weather

API_BASE_URL = ""  # leave empty: the environment, a Colab secret, or a prompt gives it

def find_api_base_url():
    if API_BASE_URL:
        return API_BASE_URL
    if os.environ.get("UTFOREST_API_BASE_URL"):
        return os.environ["UTFOREST_API_BASE_URL"]
    try:
        from google.colab import userdata  # Google Colab only
        return userdata.get("UTFOREST_API_BASE_URL")
    except Exception:
        pass
    from getpass import getpass
    return getpass("API base URL: ")

UTForestDataFetcher.configure(base_url=find_api_base_url())
print("UTForestDataFetcher", UTForestDataFetcher.__version__)

## 2. 一覧（Catalog）

指定できる演習林、観測地点、観測項目と、データのある期間です。ここに出る名前を、以下の `forest`、`site`、`filter` に使います。

In [ ]:
Weather.Catalog(forest="千葉演習林", site="札郷")

## 3. DOI の解決（DoiResolver）

期間の各月について、版番号付きの DOI を 1 つずつ返します。既定は、今日の時点で最新の版です。

In [ ]:
FOREST, SITE = "千葉演習林", "札郷"
RANGE = ("2025-01-01", "2025-12-31")

dois = Weather.DoiResolver(forest=FOREST, site=SITE, range=RANGE)
dois

In [ ]:
# 版番号なしの DOI（版を問わない引用やリンク用。FetchData には渡せない）
Weather.DoiResolver(forest=FOREST, site=SITE, range=RANGE, version="canonical")[:3]

## 4. データの取得と可視化

`FetchData` は DOI の集合をそのまま受け取るので、どの版から作ったかが固定されます。日付で絞るときは pandas の機能を使います。

In [ ]:
df = Weather.FetchData(dois=dois, filter=["平均気温", "最高気温", "最低気温", "降水量"])
print(len(df), "days from", len(df.attrs["dois"]), "DOIs")
df.loc["2025-07-01":"2025-07-07"]

In [ ]:
import matplotlib.pyplot as plt

# The default fonts have no Japanese glyphs, so the plot uses English labels.
labels = {"平均気温": "mean", "最高気温": "max", "最低気温": "min"}
code = Weather.Catalog(forest=FOREST, site=SITE)["site_code"].iloc[0]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7))
df[["最高気温", "平均気温", "最低気温"]].rename(columns=labels).plot(ax=ax1, xlabel="date", ylabel="°C", title=f"{code} 2025: air temperature")
monthly = df["降水量"].resample("MS").sum()
monthly.index = monthly.index.strftime("%Y-%m")
monthly.plot(ax=ax2, kind="bar", xlabel="month", ylabel="mm", title="precipitation by month", rot=45)
plt.tight_layout()

## 5. 版と更新の確認

`as_of` を指定すると、その日に最新だった版の DOI を返します。論文を書いた日（ここでは 2026-03-01）の版を再現するときに使います。`CheckUpdates` は、その後に新しい版が出た月を返します。

In [ ]:
written = Weather.DoiResolver(forest=FOREST, site=SITE, range=RANGE, as_of="2026-03-01")
Weather.CheckUpdates(dois=written)

In [ ]:
# 新しい版で何が変わったか
import pandas as pd

updates = Weather.CheckUpdates(dois=written)
if not updates.empty:
    row = updates.iloc[0]
    old = Weather.FetchData(dois=[row["doi"]])
    new = Weather.FetchData(dois=[row["latest"]])
    changed = (old != new).any(axis=1)
    display(pd.concat({"old": old[changed], "new": new[changed]}, axis=1))

## 6. 引用（Cite）

doi.org の content negotiation で引用文を作ります。mock の DOI は doi.org にないので、今は引用できません（JaLC と契約して本物の DOI になったら使えます）。

In [ ]:
try:
    print(Weather.Cite(dois=dois[:2], format="bibtex"))
except ValueError as error:
    print(error)